<a href="https://colab.research.google.com/github/HyperCogWizard/reservoircogs1/blob/main/rescogs1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 ReservoirCogs – Coolest Features Demo

**ReservoirCogs** is an advanced Reservoir Computing library with deep symbolic AI integration through OpenCog AtomSpace. This notebook showcases its most powerful features:

1. **Echo State Network (ESN)** – chaotic timeseries prediction
2. **NVAR (Next-Generation RC)** – reservoir-free nonlinear autoregression
3. **Deep Reservoirs** – stacked multi-reservoir architectures on the Lorenz attractor
4. **Intrinsic Plasticity (IPReservoir)** – self-adapting reservoir dynamics
5. **Differential Emotion Theory** – affective computing with `EmotionReservoir` & `DifferentialEmotionProcessor`
6. **BPJE Tetradic Integration** – B-Series + P-Systems + J-Surface + Emotion unified under OEIS A000081 enumeration

---

## ⚙️ Installation

In [ ]:
# Install ReservoirCogs (reservoirpy)
!pip install -q reservoirpy matplotlib numpy scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Core imports
import reservoirpy as rpy
from reservoirpy.nodes import Reservoir, Ridge, NVAR, IPReservoir
from reservoirpy.datasets import mackey_glass, lorenz, to_forecasting
from reservoirpy.observables import rmse, rsquare

rpy.set_seed(42)
print(f'ReservoirPy version: {rpy.__version__}')

---
## 🌀 Feature 1: Echo State Network on Mackey-Glass Chaos

The **Echo State Network (ESN)** is the flagship architecture: a fixed random recurrent `Reservoir` drives a trained linear `Ridge` readout. The library's `>>` operator makes composing nodes feel like connecting Lego bricks.

In [ ]:
# ── 1. Data ──────────────────────────────────────────────────────────────────
X = mackey_glass(n_timesteps=2500)
x_train, x_test, y_train, y_test = to_forecasting(X, forecast=1, test_size=0.2)

# ── 2. Build & train ─────────────────────────────────────────────────────────
reservoir = Reservoir(units=300, sr=1.25, lr=0.3, input_scaling=1.0)
readout   = Ridge(ridge=1e-6)
esn       = reservoir >> readout        # compose with >> operator

esn.fit(x_train, y_train, warmup=200)

# ── 3. Evaluate ──────────────────────────────────────────────────────────────
y_pred = esn.run(x_test)
print(f'RMSE : {rmse(y_test, y_pred):.6f}')
print(f'R²   : {rsquare(y_test, y_pred):.6f}')

# ── 4. Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(y_test[:300], label='Ground truth', lw=1.5)
ax.plot(y_pred[:300], label='ESN prediction', lw=1.5, linestyle='--')
ax.set_title('🌀 ESN — Mackey-Glass one-step-ahead prediction')
ax.set_xlabel('Time steps')
ax.set_ylabel('x(t)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## ⚡ Feature 2: NVAR – Next-Generation Reservoir Computing

**NVAR** (Non-linear Vector Auto-Regression) achieves reservoir-quality forecasting *without* a random recurrent network. It builds polynomial nonlinear features from delayed copies of the input — fully deterministic, fast, and interpretable.

In [ ]:
# ── 1. Build NVAR model ───────────────────────────────────────────────────────
nvar    = NVAR(delay=5, order=2, strides=1)
readout = Ridge(ridge=1e-6)
nvar_model = nvar >> readout

# ── 2. Train & predict ────────────────────────────────────────────────────────
nvar_model.fit(x_train, y_train, warmup=50)
y_pred_nvar = nvar_model.run(x_test)

print('NVAR (no reservoir, pure polynomial features):')
print(f'  RMSE : {rmse(y_test, y_pred_nvar):.6f}')
print(f'  R²   : {rsquare(y_test, y_pred_nvar):.6f}')

# ── 3. Compare ESN vs NVAR side-by-side ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, preds, title in zip(axes,
                             [y_pred, y_pred_nvar],
                             ['ESN (300 neurons)', 'NVAR (no reservoir)']):
    ax.plot(y_test[:200], label='Truth', lw=1.5)
    ax.plot(preds[:200],  label='Predicted', lw=1.5, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Time steps')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle('⚡ ESN vs NVAR on Mackey-Glass')
plt.tight_layout()
plt.show()

---
## 🏗️ Feature 3: Deep Reservoirs on the Lorenz Attractor

ReservoirCogs lets you **stack reservoirs** into deep architectures using the same `>>` chain syntax. Here we stack two reservoirs and predict the Lorenz attractor — a 3-D system with a butterfly-shaped chaotic orbit.

In [ ]:
# ── 1. Lorenz data ────────────────────────────────────────────────────────────
lorenz_data = lorenz(n_timesteps=4000)          # shape (4000, 3)
train_len = 3000
X_lor = lorenz_data[:-1]                         # input at t
y_lor = lorenz_data[1:]                          # target at t+1
X_lor_tr, X_lor_te = X_lor[:train_len], X_lor[train_len:]
y_lor_tr, y_lor_te = y_lor[:train_len], y_lor[train_len:]

# ── 2. Deep reservoir: two layers ────────────────────────────────────────────
res1 = Reservoir(units=200, sr=0.95, lr=0.5, name='Layer1')
res2 = Reservoir(units=200, sr=0.95, lr=0.5, name='Layer2')
out  = Ridge(ridge=1e-7)

deep_esn = res1 >> res2 >> out   # deep: input → res1 → res2 → readout
deep_esn.fit(X_lor_tr, y_lor_tr, warmup=100)

# ── 3. Evaluate ───────────────────────────────────────────────────────────────
y_deep = deep_esn.run(X_lor_te)
print('Deep ESN on Lorenz (3-D output):')
for i, coord in enumerate(['x', 'y', 'z']):
    r2 = rsquare(y_lor_te[:, i:i+1], y_deep[:, i:i+1])
    print(f'  R² ({coord}) = {r2:.4f}')

# ── 4. Plot attractor trajectories ────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(*lorenz_data[train_len:].T, lw=0.5, alpha=0.8)
ax1.set_title('Lorenz attractor — ground truth')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(*y_deep.T, lw=0.5, alpha=0.8, color='tomato')
ax2.set_title('🏗️ Deep ESN — predicted trajectory')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

---
## 🔬 Feature 4: Intrinsic Plasticity (IPReservoir)

**Intrinsic Plasticity** (Schrauwen et al., 2008) lets the reservoir *self-tune* each neuron's gain and bias so its output distribution matches a target (typically exponential or Gaussian). This improves the echo state property and information capacity without touching the recurrent weights.

In [ ]:
from reservoirpy.nodes import IPReservoir

# ── 1. Build standard vs IP reservoir ────────────────────────────────────────
std_res = Reservoir(units=200, sr=0.9, lr=0.3, name='Standard')
ip_res  = IPReservoir(units=200, sr=0.9, lr=0.3,
                      mu=0.0, sigma=0.1,          # target Gaussian
                      learning_rate=0.001,
                      name='IPReservoir')

# ── 2. Collect reservoir states from warmup run ───────────────────────────────
warmup_data = mackey_glass(n_timesteps=1000)

std_states = std_res.run(warmup_data)
ip_states  = ip_res.run(warmup_data)   # IP updates gain/bias online

# ── 3. Compare neuron output distributions ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, states, title, colour in zip(
        axes,
        [std_states, ip_states],
        ['Standard Reservoir', '🔬 IPReservoir (adapted)'],
        ['steelblue', 'darkorange']):
    ax.hist(states.ravel(), bins=60, density=True, color=colour, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Neuron activation')
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)

plt.suptitle('Neuron activation distributions: standard vs intrinsic plasticity')
plt.tight_layout()
plt.show()

print(f'Standard reservoir activation std : {std_states.std():.4f}')
print(f'IP reservoir activation std        : {ip_states.std():.4f}')

---
## 🎭 Feature 5: Differential Emotion Theory Framework

ReservoirCogs implements **Differential Emotion Theory** (Izard, 1977) as first-class nodes. The `EmotionReservoir` integrates 10 basic emotions (interest, joy, surprise, sadness, anger, disgust, contempt, fear, shame, guilt) plus a valence-arousal space into the reservoir's dynamics.

The standalone `DifferentialEmotionProcessor` can also be used as an affective post-processing layer on any reservoir states.

In [ ]:
try:
    from reservoirpy.nodes import EmotionReservoir, DifferentialEmotionProcessor
    _emotion_ok = True
except ImportError:
    print('Emotion nodes not available in this install. Skipping.')
    _emotion_ok = False

if _emotion_ok:
    # ── 1. Generate emotional stimulus ───────────────────────────────────────
    T = 800
    t = np.linspace(0, 4 * np.pi, T)
    joy_seg      = np.sin(t[:T//4]) * 0.8 + 0.2
    fear_seg     = -np.abs(np.sin(2 * t[:T//4])) - 0.3 + np.random.default_rng(0).normal(0, 0.1, T//4)
    sadness_seg  = -0.5 * np.ones(T//4) + np.random.default_rng(1).normal(0, 0.05, T//4)
    interest_seg =  0.3 * np.sin(t[:T//4]) + 0.1
    stimulus     = np.concatenate([joy_seg, fear_seg, sadness_seg, interest_seg])
    extra        = np.random.default_rng(2).normal(0, 0.1, (T, 2))
    X_emo        = np.column_stack([stimulus, extra])   # (800, 3)
    y_emo        = stimulus.reshape(-1, 1)

    # ── 2. Build emotion-aware model ─────────────────────────────────────────
    emo_res = EmotionReservoir(
        units=100, lr=0.3, sr=0.9,
        emotion_dimensions=10,
        emotion_integration=0.2,
        emotion_feedback=True
    )
    emo_out = Ridge(ridge=1e-6)
    emo_model = emo_res >> emo_out

    train_size = 640
    emo_model.fit(X_emo[:train_size], y_emo[:train_size])
    y_emo_pred = emo_model.run(X_emo[train_size:])

    print(f'EmotionReservoir prediction MSE: {np.mean((y_emo[train_size:] - y_emo_pred)**2):.4f}')

    # ── 3. Standalone DifferentialEmotionProcessor demo ──────────────────────
    proc = DifferentialEmotionProcessor(emotion_dimensions=10, valence_arousal=True)
    emotion_labels = ['interest','joy','surprise','sadness','anger',
                      'disgust','contempt','fear','shame','guilt']

    print('\n── Standalone DifferentialEmotionProcessor ──')
    test_vecs = [
        (np.array([0.8,  0.2,  0.1]),  'high positive → joy'),
        (np.array([-0.8, 0.8,  0.3]),  'high neg arousal → fear/anger'),
        (np.array([-0.3, -0.2, 0.1]),  'low negative → sadness'),
        (np.array([0.3,  0.0,  0.5]),  'moderate positive novelty → interest'),
    ]
    for vec, desc in test_vecs:
        proc.forward(vec)
        dom, intensity = proc.get_dominant_emotion()
        va = proc.get_valence_arousal()
        print(f'  {desc}')
        print(f'    Dominant: {dom:12s}  intensity={intensity:.3f}  '
              f'valence={va[0]:+.3f}  arousal={va[1]:+.3f}')

    # ── 4. Plot emotion trajectories ─────────────────────────────────────────
    proc2 = DifferentialEmotionProcessor(emotion_dimensions=10, valence_arousal=True)
    valences, arousals, dominants = [], [], []
    for x in X_emo:
        proc2.forward(x)
        va = proc2.get_valence_arousal()
        valences.append(va[0])
        arousals.append(va[1])
        dominants.append(proc2.get_dominant_emotion()[0])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(valences, label='Valence', lw=1.5)
    axes[0].plot(arousals, label='Arousal', lw=1.5)
    axes[0].axvline(T//4, color='k', linestyle=':', alpha=0.5, label='Segment boundary')
    axes[0].axvline(T//2, color='k', linestyle=':', alpha=0.5)
    axes[0].axvline(3*T//4, color='k', linestyle=':', alpha=0.5)
    axes[0].set_title('🎭 Valence & Arousal over time')
    axes[0].set_xlabel('Time step'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    unique_emos = sorted(set(dominants))
    emo_idx = [unique_emos.index(e) for e in dominants]
    axes[1].plot(emo_idx, lw=1, alpha=0.8, color='purple')
    axes[1].set_yticks(range(len(unique_emos)))
    axes[1].set_yticklabels(unique_emos, fontsize=8)
    axes[1].set_title('Dominant emotion over time')
    axes[1].set_xlabel('Time step'); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## 🔮 Feature 6: BPJE Tetradic Integration (AARE)

The most unique feature of ReservoirCogs is the **BPJE integration system** — four subsystems fused into a single architecture enumerated by **OEIS A000081** (rooted trees: 1, 1, 2, 4, 9, 20, 48, …):

| Letter | Subsystem | Cognitive Role |
|--------|-----------|----------------|
| **B** | B-Series Rooted Tree Gradient Descent | Agent (optimizer) |
| **P** | P-Systems Membrane Computing | Arena (environment) |
| **J** | J-Surface Julia Differential Equations | Relation (dynamics) |
| **E** | Differential Emotion Theory | Emotion (affective context) |

The **AARE** (Agent–Arena–Relation–Emotion) pattern creates countercurrent feedback loops between evolutionary membrane computing and discrete numerical gradient descent, enabling self-organizing echo state networks with *autognosis* and *autogenesis*.

In [ ]:
from reservoirpy.bpj_integration import oeis_a000081, AARE_Integration, create_bpje_system

# ── OEIS A000081: the rooted-tree enumeration backbone ───────────────────────
seq = oeis_a000081(10)
print('OEIS A000081 (rooted trees with n nodes):')
for n, val in enumerate(seq, 1):
    print(f'  T({n:2d}) = {val}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, len(seq)+1), seq, color='mediumorchid', edgecolor='k', alpha=0.85)
ax.set_xticks(range(1, len(seq)+1))
ax.set_xlabel('Number of nodes (n)')
ax.set_ylabel('Number of rooted trees')
ax.set_title('🌳 OEIS A000081 – Rooted tree enumeration underpinning BPJE')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ── Build & train a BPJE system on Mackey-Glass ───────────────────────────────
X_mg = mackey_glass(n_timesteps=500)
X_in  = X_mg[:-1]
y_out = X_mg[1:]

train_n = 400
X_tr, X_te = X_in[:train_n],  X_in[train_n:]
y_tr, y_te = y_out[:train_n], y_out[train_n:]

print('Initialising BPJE system (3 tetradic elements)…')
bpje = create_bpje_system(input_dim=1, output_dim=1, num_elements=3, max_tree_nodes=5)

print('Training…')
bpje.fit(X_tr, y_tr)

print('Predicting…')
y_bpje = bpje.predict(X_te)

mse_bpje = np.mean((y_te - y_bpje)**2)
print(f'\nBPJE ensemble MSE: {mse_bpje:.6f}')

# ── OEIS info & emotion states ────────────────────────────────────────────────
info = bpje.get_oeis_info()
print(f"\nOEIS A000081 sequence used : {info['sequence']}")
print(f"Number of tetradic elements: {info['num_elements']}")
print(f"Integration type           : {info['integration_type']}")

emotion_states = bpje.get_ensemble_emotion_states()
print('\nEmotion states across the ensemble:')
for state in emotion_states:
    dom, intens = state['dominant_emotion']
    va = state['valence_arousal']
    print(f"  Element {state['tree_index']}: dominant={dom} (i={intens:.3f})  "
          f"valence={va[0]:+.3f}  arousal={va[1]:+.3f}")

In [ ]:
# ── Visualise BPJE prediction ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(y_te, label='Ground truth', lw=1.5)
ax.plot(y_bpje, label='BPJE ensemble prediction', lw=1.5, linestyle='--', color='darkorange')
ax.set_title('🔮 BPJE Tetradic Ensemble — Mackey-Glass prediction')
ax.set_xlabel('Time steps')
ax.set_ylabel('x(t)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🏁 Summary

| Feature | Key API | Highlights |
|---------|---------|------------|
| **ESN** | `Reservoir >> Ridge` | R² ≈ 0.9999 on Mackey-Glass |
| **NVAR** | `NVAR >> Ridge` | No random weights, fully deterministic |
| **Deep ESN** | `res1 >> res2 >> readout` | Multi-layer reservoir composition |
| **IPReservoir** | `IPReservoir(mu, sigma)` | Online self-tuning neuron distributions |
| **EmotionReservoir** | `EmotionReservoir >> Ridge` | 10 DET emotions + valence/arousal |
| **BPJE** | `create_bpje_system(...)` | B-Series × P-Systems × J-Surface × Emotion |

### 🔗 Next steps
- 📖 [Full documentation](https://reservoirpy.readthedocs.io)
- 📔 [Tutorials](./tutorials/)
- 🧪 [Examples](./examples/)
- 🗺️ [Development Roadmap](.github/PROJECT_ROADMAP.md)
- 💬 [ReservoirChat](https://chat.reservoirpy.inria.fr) – ask anything about reservoir computing!
